In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # Load environment variables from .env file
openai_client = OpenAI()  # Initialize the OpenAI client

In [ ]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [ ]:
from rag_helper import RAGBase

instrcutions = '''
You are a course teaching assistant. 
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
'''.strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instrcutions,
    )

In [ ]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system:\n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and start a model locally:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3. To test that the local server is running:\n   ```bash\n   curl http://localhost:11434\n   ```\n\nYou should get a response like:\n```json\n{"models": [...]}\n```\n\n4. If you want to use it from Python:\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": your_prompt}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```'

In [ ]:
messages = [
    {"role": 'user', "content": "How do I run Olama?",},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    )

response.output_text

'If you mean **Ollama** (often misspelled “Olama”), here’s the quick way to run it:\n\n## 1) Install Ollama\n- **macOS / Windows:** download it from the official site: **https://ollama.com**\n- **Linux:** use the install command from the Ollama website\n\n## 2) Start the service\nUsually it starts automatically after install, but you can run:\n\n```bash\nollama serve\n```\n\n## 3) Run a model\nFor example:\n\n```bash\nollama run llama3\n```\n\nThat will download the model if needed and open an interactive chat.\n\n## 4) Check what models are installed\n```bash\nollama list\n```\n\n## 5) Run Ollama from code or API\nBy default it serves an API on:\n\n```bash\nhttp://localhost:11434\n```\n\nExample curl request:\n\n```bash\ncurl http://localhost:11434/api/generate -d \'{\n  "model": "llama3",\n  "prompt": "Write a haiku about computers."\n}\'\n```\n\nIf you want, I can also show you:\n- how to install Ollama on your OS,\n- how to run a specific model,\n- or how to use it from Python.'

In [ ]:
def search(query):
    boost_dict = {'question':3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [ ]:
search("How do I run Ollama locally?")

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    'description': "Search the FAQ database for entries matching the given query.",
    'parameters':{
        "type": "object",
        "properties":{
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)


call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"Olama run install start Ollama"}', call_id='call_pYeb3qTWobiFG1AbLzEGEp5d', name='search', type='function_call', id='fc_04603a5381e21874006a31c612975c81a1bb06f751fbec111e', namespace=None, status='completed')

In [ ]:
call.arguments

'{"query":"Olama run install start Ollama"}'

In [ ]:
import json

args = json.loads(call.arguments)


In [ ]:
results = search(**args)
results

[{'id': '1d0b969028',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Ollama: How to install Ollama?',
  'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip install ollama\n```\n\n

In [ ]:
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "1d0b969028",
    "course": "llm-zoomcamp",
    "section": "Module 1: RAG",
    "question": "Ollama: How to install Ollama?",
    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{\"models\": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash\npip insta

In [ ]:
function_call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
}

In [ ]:
messages.append(call)
messages.append(function_call_output)

messages

[{'role': 'user', 'content': 'How do I run Olama?'},
 ResponseFunctionToolCall(arguments='{"query":"Olama run install start Ollama"}', call_id='call_pYeb3qTWobiFG1AbLzEGEp5d', name='search', type='function_call', id='fc_04603a5381e21874006a31c612975c81a1bb06f751fbec111e', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_pYeb3qTWobiFG1AbLzEGEp5d',
  'output': '[\n  {\n    "id": "1d0b969028",\n    "course": "llm-zoomcamp",\n    "section": "Module 1: RAG",\n    "question": "Ollama: How to install Ollama?",\n    "answer": "First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\\n\\n- **macOS**: Download the `.pkg` and install it.\\n- **Windows**: Download the `.msi` and install it.\\n- **Linux**: Run the following command in the terminal:\\n\\n  ```bash\\n  curl -fsSL https://ollama.com/install.sh | sh\\n  ```\\n\\nOnce installed, open a terminal and type:\\n\\n```bash\\nollam

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [ ]:
print(response.output_text)

To run **Ollama**:

1. **Install it** from https://ollama.com/download  
   - macOS: download the `.pkg`
   - Windows: download the `.msi`
   - Linux: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Start a model** in your terminal:
```bash
ollama run llama3
```

That will download the model if needed and open a chat interface.

If you want, you can also test the local server with:
```bash
curl http://localhost:11434
```

If you meant **“Ollama serve”** or got a connection error, tell me your OS and I’ll help you start it properly.


In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = 'I just discovered the course. Can I join it?'

messages = [
    {'role': 'developer', 'content': instructions},
    {"role": "user", "content": question},
]

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)


In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

In [ ]:
messages.extend(response.output)

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment"}
function_call: search {"query":"new student join the course enrollment late join discovered course"}


In [ ]:
messages

[{'role': 'developer',
  'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore."},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enrollment"}', call_id='call_V65RkWFUFiZDwqn6VQq0ePFW', name='search', type='function_call', id='fc_0b5ec64e5c39d63b006a31c6170460819db2692c6c93f94f27', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student join the course enrollment late join discovered course"}', call_id='call_eYzp8Kj

In [ ]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

print(response.output_text)

Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still being accepted. If you’d like, I can also help you with how to start the course or what to do next. Are there other areas you want to explore?


In [ ]:
messages = [
    {'role': 'developer', 'content': instructions},
    {"role": "user", "content": question},
]

it = 1

while True:
    print(f"Iteration {it}")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    if not has_function_calls:
        break

Iteration 1
function_call: search {"query":"join course discovered the course can I join enrollment late registration join course FAQ"}
function_call: search {"query":"new student join the course enrollment FAQ discovered course"}
Iteration 1
ASSISTANT:
Yes, you can still join. You can start learning and working through the materials even if you just discovered it.

One important note: if you want a certificate, you need to submit your project while the course is still accepting submissions. Also, certificates are only available for the live cohort, not self-paced.

If you want, I can also help you with how to start the course or explain the weekly workflow.
